
# Handoff パターン
**2つの Handoff パターン**が実装されています：

| # | パターン | モード | 構成 | セル |
|---|---|---|---|---|
| 1 | **調査・要約ワークフロー** | autonomous | `coordinator` → `research_agent` / `summary_agent` | 10-18 |
| 2 | **カスタマーサポート（対話型）** | human-in-the-loop | `triage` → `order_specialist` / `return_specialist` / `billing_specialist`（`with_autonomous_mode()` なし） | 20-26 |

本質的には **2つの軸の組み合わせ** です：

- **エージェント構成**: 調査・要約 vs カスタマーサポート
- **実行モード**: autonomous（自律完結）vs human-in-the-loop（ユーザー入力待ち）

```
              autonomous            human-in-the-loop
調査・要約     パターン1              （このノートブックでは未実装）
サポート       （このノートブックでは未実装）  パターン2
```

In [ ]:
import logging  
from typing import Any, List  
import os  
from dotenv import load_dotenv  

from agent_framework.azure import AzureOpenAIChatClient
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.orchestrations import HandoffBuilder

# import warnings
# warnings.simplefilter('ignore')

load_dotenv(override=True)

## OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# 環境変数ベースで設定（Agent Framework は標準の OTEL_* を読む）
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Metrics は無効化（必要に応じて変更）

# enable_sensitive_data=True を指定して機微データ(OpenAI の IN/OUT データ)の収集を有効化
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

In [ ]:
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Tools の定義(MCP Streamable HTTP)

In [ ]:
# MCP Streamable HTTP ツールを作成（初回接続が重い場合があるので timeout を長めにする）
mcp_tool = MCPStreamableHTTPTool(
    name="mcp_tools",
    url=mcp_server_uri,
    description="ゲームショップの製品、注文、在庫、ユーザー情報、Twitter分析データを取得するためのツールです。",
    timeout=120,
)

mcp_microsoft_learn_tool = MCPStreamableHTTPTool(
    name="mcp_microsoft_learn_tools",
    url="https://learn.microsoft.com/api/mcp",
    description="Microsoft Learn Docs を取得するためのツールです。Microsoft 公式ドキュメントから最新の情報を提供します。",
    timeout=120,
)

# Azure OpenAI クライアントの作成
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

print(f"MCP Tool: {mcp_tool}")

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "Handoff",
    print_final: bool = True,
) -> list[Message]:
    """
    
    workflow.run(task, stream=True) を実行し、ストリーミング出力を
    Jupyter Notebook 上でリアルタイムに表示するヘルパー。

    前提（GroupChatBuilder 使用時）:
      GroupChatBuilder(..., intermediate_outputs=True) でビルドすること。
      デフォルト(False)ではオーケストレーターの output のみが yield され、
      参加者の AgentResponseUpdate はフィルタされる。

    表示:
      - AgentResponseUpdate → トークンを逐次表示（sys.stdout.write + flush）
      - executor 切替時 → 改行 + 名前ヘッダ
      - 最終 list[Message] → CONVERSATION LOG としてまとめ表示

    注意:
      ★ start_as_current_span() を使用し、ワークフロー内部のスパンを
      このスパンの子として正しくネストさせる。
      _process_events は通常の async 関数であり async generator ではないため、
      コンテキストマネージャと組み合わせても GeneratorExit 問題は発生しない。
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Jupyter でも確実にフラッシュする書き込み"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- executor 切替通知 -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"🤖 {eid}: ")
                    last_executor_id = eid

            # ----- output イベント -----
            elif event.type == "output":
                data = event.data

                # (1) ストリーミングチャンク: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) 非ストリーミング応答: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) 最終会話ログ: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # ★ start_as_current_span() を使用してワークフロー内部スパンを子としてネスト。
    #   _process_events は通常の async 関数なのでコンテキスト競合は発生しない。
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # ストリーム末尾の改行
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    
    Args:
        workflow: 可視化するワークフローオブジェクト
        filename: 保存するファイル名（拡張子なし）
        show_preview: プレビューを表示するかどうか
    
    Returns:
        保存されたSVGファイルのパス
    """
    # WorkflowVizオブジェクトを作成
    viz = WorkflowViz(workflow)
    
    # 3. SVGファイルとして保存
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVGファイルを保存しました: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. SVGをプレビュー表示
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ エラー: 'graphviz'パッケージがインストールされていません")
        print("インストール方法: pip install agent-framework[viz] --pre")
        print(f"詳細: {e}")
        return None
    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        return None

## エージェント定義
### Handoff パラメーター
Handoff の場合、各エージェントが他のどのエージェントに委譲できるかを指定します。Selector のような遷移先を決めるマネージャーが存在しないため、遷移先の決定は各エージェントのプロンプトに依存します。

In [ ]:
# 3. -----------------  Agent Definitions -----------------  
coordinator = Agent(
    name="coordinator",
    client=chat_client,
    description="タスクを計画するエージェント。ユーザーのリクエストを適切な専門エージェントに振り分けてください。",
    instructions=(
"""
あなたはコーディネーターです。ユーザークエリを調査タスクと要約タスクに分解します。
二つのタスクを適切な専門家に順番に割り当てます。
すべての調査および要約が完了したら、FINAL_RESPONSE としてユーザーに返答してください。
"""
    )
)

research_agent = Agent(
    name="research_agent",
    client=chat_client,
    description="research_agent",
    tools=[mcp_tool, mcp_microsoft_learn_tool],
    instructions=(
"""
あなたはMicrosoft Learnサイトでトピックを徹底的に調査するリサーチスペシャリストです
- 調査課題を与えられたら、それを複数の側面に分解し、それぞれを探求してください
- 複数の回答にまたがって調査を継続してください。1回の回答で全てを終わらせようとしないでください
- 各回答の後、他に探求すべき点がないか考えてください
- トピックを包括的にカバーしたら（少なくとも3～4つの異なる側面を網羅）
- 回答の最初に【🤖調査】を挿入します。
- 完了したら「coordinator」に進行を戻してください。各回答は一つの側面に集中させてください。
"""
    ),
)

summary_agent = Agent(
    name="summary_agent",
    client=chat_client,
    description="summary_agent",
    tools=[mcp_tool],
    instructions=(
"""            
リサーチ結果を要約してください。簡潔で整理された要約を提供してください。
- 回答の最初に【🤖要約】を挿入します。
- 要約は最後に1回だけ行ってください。
- 完了したら、「coordinator」へ進行を戻してください。
"""
    ),
)

In [ ]:
# 4. -----------------  Assemble Team -----------------  
# HandoffBuilder を使用してワークフローを構築
workflow = (
    HandoffBuilder(
        name="autonomous_iteration_handoff",
        participants=[coordinator, research_agent, summary_agent],
    )
    .with_start_agent(coordinator)
    .add_handoff(coordinator, [research_agent, summary_agent])
    .add_handoff(research_agent, [coordinator])  # research_agent はコーディネーターに返却可能
    .add_handoff(summary_agent, [coordinator])
    # with_autonomous_mode: エージェントが自律的にタスクを処理するモード
    # turn_limits: エージェントごとの自律ターン数制限（無限ループ防止の安全装置）
    .with_autonomous_mode(turn_limits={"coordinator": 5, "research_agent": 5, "summary_agent": 5})
    # with_termination_condition: タスク完了の意味的な判断（理想的な終了条件）
    # コーディネーターが3回応答 = 調査と要約が完了したと判断
    .with_termination_condition(
        lambda conv: sum(1 for msg in conv if msg.author_name == "coordinator" and msg.role == "assistant")
        >= 3
    )
    .build()
)

In [ ]:
# ワークフローの可視化
svg_path = visualize_workflow(
    workflow, 
    filename="handoff_workflow",
    show_preview=True
)


- `with_autonomous_mode()`:ワークフローを自律モードに設定します。自律モードでは、エージェント（スペシャリストを含む）は、明示的にハンドオフツールを呼び出すか、ターン制限に達するまでタスクを反復処理し続けます。これにより、スペシャリストは長期にわたる自律タスク（調査、コーディング、分析など）を実行でき、コーディネーターやユーザーに制御を早期に返す必要がありません。`with_autonomous_mode()` を呼ばない場合はデフォルトで human-in-the-loop モード（エージェントの応答後にユーザー入力を待つ）になります。

  - `turn_limits`: エージェントごとの自律ターン数上限を辞書で指定（例: `{"agent_name": 5}`）。無限ループ防止の安全装置として機能します。


## Handoff の実行

In [ ]:
# task = "ユーザーID:123 の出荷状況を確認してください。"
task = "Azure AI Search の 2025年の新機能について3つ簡潔に教えてください。"
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer, span_name="Handoff")

In [ ]:
for msg in final_conversation:
    print(f"⭐️Role: {msg.role}, 👤Author: {msg.author_name}, 📝Text: {msg.text[:100]}...")

In [ ]:
for msg in final_conversation:
    print(msg.author_name)

このセルは **3つのエージェントによる自律型ハンドオフワークフローの組み立て** を行っています。

本質的にやっていることは:

1. **エージェントの遷移ルート定義** — 誰が誰に仕事を渡せるかのグラフ構造を構築
   - `coordinator` → `research_agent` / `summary_agent`（調査・要約を委譲）
   - `research_agent` → `coordinator`（結果を返却）
   - `summary_agent` → `coordinator`（結果を返却）

2. **自律モード設定** — ユーザー入力なしでエージェント間を自動で遷移させる。各エージェント最大5ターンで打ち切り

3. **終了条件** — `coordinator` が3回 assistant として応答したら完了と判断（＝調査の指示→調査結果受領→要約結果受領のサイクルが回った）

要するに、**「coordinator が司令塔となり、research → summary の順に仕事を委譲し、結果が揃ったら自動終了する」** というオーケストレーションの設計図です。

```
ユーザー入力
  ↓
coordinator (計画・振り分け)  ←──┐
  ↓ handoff                     │
research_agent (調査) ──────────┤
  ↓ (coordinator経由で再委譲)    │
summary_agent (要約) ───────────┘
  ↓
coordinator (最終応答) → 終了
```

In [ ]:
# ========================================
# 1. エージェント定義 - シンプル版
# ========================================

# トリアージエージェント: 問い合わせを適切な専門家に振り分ける
triage = Agent(
    name="triage",
    client=chat_client,
    instructions="""
あなたはカスタマーサポートのトリアージ担当です。
顧客の問い合わせ内容に応じて、適切な専門家に振り分けてください：

- 注文状況や配送に関する問い合わせ → order_specialist
- 返品や交換に関する問い合わせ → return_specialist
- 請求や支払いに関する問い合わせ → billing_specialist

振り分けは handoff ツールを使って即座に行ってください。説明は不要です。
""",
)

# 注文専門家
order_specialist = Agent(
    name="order_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="""
あなたは注文・配送の専門家です。
必ず mcp_tools を使ってデータ確認してから回答してください。推測回答は禁止です。

- ユーザーが注文番号(order_id)を言った場合: まず get_shipping_status_by_order_id を呼ぶ
- ユーザーIDしか無い場合: get_shipping_status を呼ぶ
- 取得結果が空のときだけ、追加情報を質問する
- ツール結果にデータがある場合は「見つかりませんでした」と言わず、結果を要約して返す
- 完了したら終了してください。
""",
)

# 返品専門家
return_specialist = Agent(
    name="return_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="返品・交換リクエストを処理してください。完了したら終了してください。",
)

# 請求専門家
billing_specialist = Agent(
    name="billing_specialist",
    client=chat_client,
    tools=[mcp_tool],
    instructions="請求や支払いに関する問題を解決してください。完了したら終了してください。",
)

---

# ユーザー入力を受け付けるHandoff - 対話型シナリオ

**目的**: ユーザーとエージェントの対話を実現

**シナリオ**:
1. ユーザーが最初の問い合わせ
2. エージェントが質問や確認を求める
3. **ユーザーが追加情報を入力**
4. エージェントが処理を続行

**モード**: `human-in-the-loop` - ユーザーの入力を待つ

In [ ]:
# ========================================
# 1. 対話型ワークフロー構築
# ========================================

interactive_workflow = (
    HandoffBuilder(
        name="interactive_handoff",
        participants=[triage, order_specialist, return_specialist, billing_specialist],
    )
    .with_start_agent(triage)
    .add_handoff(triage, [order_specialist, return_specialist, billing_specialist])
    .add_handoff(order_specialist, [triage])
    .add_handoff(return_specialist, [triage])
    .add_handoff(billing_specialist, [triage])
    
    # human_in_loop モード: with_autonomous_mode() を呼ばないことでデフォルトの対話モードになる
    # エージェントが応答した後、ユーザー入力を待つ
    
    # 終了条件: ユーザーが "完了" または "終了" と入力したら終了
    .with_termination_condition(
        lambda conv: any(
            msg.role == "user" and msg.text and ("完了" in msg.text or "終了" in msg.text or "ありがとう" in msg.text)
            for msg in conv
        )
    )
    .build()
)

print("✅ 対話型Handoffワークフローを構築しました")

In [ ]:
# ワークフローの可視化
svg_path = visualize_workflow(
    interactive_workflow, 
    filename="interactive_handoff_workflow",
    show_preview=True
)

In [ ]:

# ========================================
# 2. ユーザー入力を受け付ける実行関数
# ========================================

import sys
from agent_framework import AgentResponseUpdate, Message, WorkflowEvent

def _as_text(data: object) -> str:
    if data is None:
        return ""
    if isinstance(data, str):
        return data
    text = getattr(data, "text", None)
    if isinstance(text, str):
        return text
    return str(data)

def _print_tool_usage_summary(conversation: list[Message]) -> None:
    tool_msgs = [
        msg for msg in conversation
        if getattr(msg, "role", None) == "tool"
    ]
    print(f"🔎 toolメッセージ数: {len(tool_msgs)}")
    if tool_msgs:
        latest = getattr(tool_msgs[-1], "text", "")
        if latest:
            preview = latest if len(latest) <= 240 else latest[:240] + "..."
            print(f"🔎 最新tool結果(先頭): {preview}")

async def run_interactive_handoff(workflow, initial_message: str):
    """
    対話型Handoff実行 - ユーザー入力を受け付ける

    ★ 重要: async for ループ内で break/return してはならない。
      workflow.run() が返す async generator を途中放棄すると、
      フレームワーク内部の _run_workflow_with_tracing() で GeneratorExit が発生し、
      OpenTelemetry のコンテキスト detach に失敗する。
      generator は必ず最後まで消費し、フラグで制御する。
    """
    print(f"\n{'='*60}")
    print(f"💬 ユーザー: {initial_message}")
    print(f"{'='*60}\n")

    # スクリプト化された入力（Notebook用）
    scripted_input = [
        "注文番号は840905です",
        "はい、お願いします",
        "ありがとうございました"
    ]
    input_index = 0

    current_agent = None
    pending_requests = []
    final_conversation: list[Message] = []

    # ★ start_as_current_span() でワークフロー内部スパンを子としてネスト
    cm = tracer.start_as_current_span("Handoff") if tracer else nullcontext()
    with cm:
        # 初回実行 — break せず最後まで消費する
        async for event in workflow.run(initial_message, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue
            if event.type == "output":
                if isinstance(event.data, AgentResponseUpdate):
                    agent_name = event.executor_id
                    if agent_name != current_agent:
                        if current_agent is not None:
                            sys.stdout.write("\n\n")
                            sys.stdout.flush()
                        sys.stdout.write(f"🤖 {agent_name}: ")
                        sys.stdout.flush()
                        current_agent = agent_name
                    chunk = event.data.text
                    if chunk:
                        sys.stdout.write(chunk)
                        sys.stdout.flush()
                elif isinstance(event.data, list):
                    final_conversation = cast(list[Message], event.data)
                    # ★ break しない — generator を最後まで消費させる
            elif event.type == "request_info":
                pending_requests.append(event)

        # final_conversation が設定済みなら完了
        if final_conversation:
            print("\n" + "="*60)
            print("✅ ワークフロー完了")
            _print_tool_usage_summary(final_conversation)
            print("="*60)
            return final_conversation

        # ユーザー入力を受け付けるループ
        while pending_requests:
            print("\n" + "-"*60)

            if input_index < len(scripted_input):
                user_input = scripted_input[input_index]
                input_index += 1
                print(f"💬 ユーザー: {user_input}")
            else:
                print("(スクリプト化された入力が終了しました)")
                break

            responses = {
                req.request_id: [Message(role="user", text=user_input)]
                for req in pending_requests
            }

            pending_requests = []
            current_agent = None

            # ★ break せず最後まで消費する
            async for event in workflow.run(stream=True, responses=responses):
                if not isinstance(event, WorkflowEvent):
                    continue
                if event.type == "output":
                    if isinstance(event.data, AgentResponseUpdate):
                        agent_name = event.executor_id
                        if agent_name != current_agent:
                            if current_agent is not None:
                                sys.stdout.write("\n\n")
                                sys.stdout.flush()
                            sys.stdout.write(f"🤖 {agent_name}: ")
                            sys.stdout.flush()
                            current_agent = agent_name
                        chunk = event.data.text
                        if chunk:
                            sys.stdout.write(chunk)
                            sys.stdout.flush()
                    elif isinstance(event.data, list):
                        final_conversation = cast(list[Message], event.data)
                elif event.type == "request_info":
                    pending_requests.append(event)

            # このラウンドで完了したか判定
            if final_conversation:
                print("\n" + "="*60)
                print("✅ ワークフロー完了")
                _print_tool_usage_summary(final_conversation)
                print("="*60)
                return final_conversation

    print("\n" + "="*60)
    print("✅ セッション終了")
    _print_tool_usage_summary(final_conversation)
    print("="*60)
    return final_conversation


## 対話型実行例

In [ ]:
# シナリオ: エージェントが追加情報を求める対話
ret = await run_interactive_handoff(
    interactive_workflow, 
    "商品の配送状況を確認したいです"
)

---

## 🎯 2つのモードの違い

### **autonomous モード** (自律型)
```python
.with_autonomous_mode(turn_limits={"agent_name": 10})
```
- エージェントが**自動的に完結**
- ユーザー入力は**最初の1回のみ**
- エージェント同士が自律的に連携

**適用例**: 
- 情報検索タスク
- データ分析
- コード生成

---

### **human-in-the-loop モード** (対話型・デフォルト)
```python
# with_autonomous_mode() を呼ばない = デフォルトで対話モード
HandoffBuilder(...)
    .with_start_agent(triage)
    .add_handoff(...)
    .build()
```
- エージェントが**ユーザーに質問**
- ユーザーが**追加情報を提供**
- 対話を繰り返して解決

**適用例**:
- カスタマーサポート
- 詳細なトラブルシューティング
- 情報収集が必要なタスク

---

### 実装のポイント

#### **autonomous モード - このノートブックの実装例**
```python
# 1回の実行で完結
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer, span_name="Handoff")
```

#### **human-in-the-loop モード - このノートブックの実装例**
```python
ret = await run_interactive_handoff(
    interactive_workflow,
    "商品の配送状況を確認したいです"
)
```

#### **human-in-the-loop モード - 入力ループの要点**
```python
while pending_requests:
    user_input = input("💬 あなた: ")  # または scripted_input[i]
    responses = {
        req.request_id: [Message(role="user", text=user_input)]
        for req in pending_requests
    }
    async for event in workflow.send_responses_streaming(responses):
        ...
```

**使い分け**: タスクの性質に応じて適切なモードを選択 🎯